In [1]:
import numpy as np
import pandas as pd
import random

In [2]:
data = pd.read_csv("/content/sample_data/mnist_train_small.csv")

In [3]:
data.head()

,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.581,0.582,0.583,0.584,0.585,0.586,0.587,0.588,0.589,0.590
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
data.shape

(19999, 785)

In [5]:
np.array(data).shape

(19999, 785)

In [6]:
data = np.array(data)
M, N = data.shape # M = 19000, N = 785
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:N]/255

data_train = data[1000:M].T
Y_train = data_train[0]
X_train = data_train[1:N]/255

In [7]:
def init_params(N1, N2, N3):
    W1 = np.random.rand(N1, N2) - 0.5
    B1 = np.random.rand(N2,1) - 0.5
    W2 = np.random.rand(N2, N3) - 0.5
    B2 = np.random.rand(N3,1) - 0.5
    return W1, B1, W2, B2

In [8]:
def ReLu(Z):
  return np.maximum(0,Z)

def deriv_ReLu(z):
    return z>0
import numpy as np

def sigmoid(z):
    """
    Computes the sigmoid activation function.
    """
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    """
    Computes the derivative of the sigmoid function with respect to its input z.

    The derivative of sigmoid(z) is sigmoid(z) * (1 - sigmoid(z)).
    """
    s = sigmoid(z)
    return s * (1 - s)

# # Example usage:
# z_value = np.array([-2, -1, 0, 1, 2])
# print("Input (z):", z_value)
# print("Sigmoid(z):", sigmoid(z_value))
# print("Sigmoid Derivative(z):", sigmoid_derivative(z_value))

# # Another example: for a single value
# single_z = 0.5
# print("\nInput (single_z):", single_z)
# print("Sigmoid(single_z):", sigmoid(single_z))
# print("Sigmoid Derivative(single_z):", sigmoid_derivative(single_z))

def softmax_columns(x):
    """
    Applies the softmax function to each column of a 2D NumPy array.
    Assumes: rows = classes, columns = samples.

    Args:
        x (numpy.ndarray): The input 2D NumPy array where rows are classes
                           and columns are samples (e.g., (num_classes, num_samples)).

    Returns:
        numpy.ndarray: The output array with softmax applied column-wise,
                       resulting in probability distributions per sample.
    """
    # Subtract max for numerical stability (important for large input values)
    # np.max(x, axis=0, keepdims=True) finds the max value in each column
    # and keeps its dimension for correct broadcasting.
    exp_x = np.exp(x - np.max(x, axis=0, keepdims=True))

    # Sum the exponentials along axis=0 (down each column),
    # meaning summing across classes for each sample.
    # keepdims=True ensures the sum result retains its dimension for broadcasting.
    sum_exp_x = np.sum(exp_x, axis=0, keepdims=True)

    # Divide each element by the sum of exponentials of its respective column.
    return exp_x / sum_exp_x

def sum_rows(matrix):
    # Sum along the rows (axis=1) using np.sum() function
    row_sums = np.sum(matrix, axis=1)
    # Reshape the resulting 1D array into a 3x1 matrix
    row_sums_3x1 = row_sums.reshape(-1, 1)
    return row_sums_3x1

def sum_columns(matrix):
    # Sum along the columns (axis=0) using np.sum() function
    column_sums = np.sum(matrix, axis=0)
    return column_sums

In [9]:
def accuracy(A2, Y):
  matrix = A2 - Y
  V1 = sum_rows(matrix)
  V2 = sum_columns(V1)
  V3 = 100*V2
  print(V3)


In [10]:
def forward_prop(X, W1, W2, B1, B2):
  Z1 = W1.T.dot(X) + B1
  A1 = ReLu(Z1)
  Z2 = W2.T.dot(A1) + B2
  A2 = sigmoid(Z2)
  return Z1, Z2, A1, A2

In [11]:
def backward_prop(A1, A2, Z1, Z2, W1, W2, B1, B2, X, Y):

  dB2 = 2*(A2-Y) * sigmoid_derivative(Z2)/N
  dW2 = A1.dot(dB2.T)/M

  dA1 = W2.dot(dB2)

  dB1 = dA1 #* deriv_ReLu(Z1)
  dW1 = X.dot(dB1.T)

  return dB1, dB2, dW1, dW2

def update_params(W1, B1, W2, B2, dB1, dB2, dW1, dW2, alpha):
    W1 = W1 - alpha * dW1
    B1 = B1 - alpha * sum_rows(dB1)
    W2 = W2 - alpha * dW2
    B2 = B2 - alpha * sum_rows(dB2)
    return W1, B1, W2, B2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

In [12]:
W1, B1, W2, B2 = init_params(784, 10, 10)

In [18]:
W1.shape, B1.shape, W2.shape, B2.shape

((784, 10), (10, 1), (10, 10), (10, 1))

In [16]:
Z1, Z2, A1, A2 = forward_prop(X_train, W1, W2, B1, B2)

In [17]:
A1.shape, A2.shape

((10, 18999), (10, 18999))

In [21]:
one_hotY = one_hot(Y_train)

In [22]:
one_hotY.shape

(10, 18999)

In [23]:
accuracy(A2, one_hotY)

[7421954.67886038]


In [ ]:
dB1, dB2, dW1, dW2 = backward_prop(A1, A2, Z1, Z2, W1, W2, B1, B2, X_train, Y_train)

In [ ]:
dB1.max(),dB2.max(),dW1.max(),dW2.max()

(np.float64(0.005728469569110103),
 np.float64(0.0003774473938934498),
 np.float64(5.700776331510746),
 np.float64(0.003012351431201569))

In [ ]:
W1, B1, W2, B2 = update_params(W1, B1, W2, B2, dB1, dB2, dW1, dW2, 0.01)

In [ ]:
def gradient_decent(W1, B1, W2, B2, X, Y, iterations, alpha):
  for i in range(iterations):
    Z1, Z2, A1, A2 = forward_prop(X, W1, W2, B1, B2)
    dB1, dB2, dW1, dW2 = backward_prop(A1, A2, Z1, Z2, W1, W2, B1, B2, X, Y)
    W1, B1, W2, B2 = update_params(W1, B1, W2, B2, dB1, dB2, dW1, dW2, alpha)
  return W1, W2, B1 ,B2

In [ ]:
W1, W2, B1 ,B2 = gradient_decent(W1, B1, W2, B2, X_train, Y_train, 10, 0.01)

In [ ]:
randomint = random.randint(0,1000)

Z1, Z2, A1, A2 = forward_prop(X_train[:,randomint].reshape(784, 1), W1, W2, B1 ,B2)

In [ ]:
Y_train[randomint]

np.int64(7)

In [ ]:
softmax_columns(A2)

array([[0.10670496],
       [0.108421  ],
       [0.10887051],
       [0.10555614],
       [0.10683545],
       [0.10847336],
       [0.10223033],
       [0.10699718],
       [0.10439195],
       [0.04151911]])

In [ ]:
A2*100

array([[3.32507518e-12],
       [7.87046531e-12],
       [7.56943218e-10],
       [4.14289913e-27],
       [1.00000000e+02],
       [1.21785123e-23],
       [3.09314775e-19],
       [6.39823647e-24],
       [3.13783128e-41],
       [1.43839063e-11]])

In [ ]:
WW1, WW2, BB1 ,BB2 = gradient_decent(784, 10, 10, X_train, Y_train, 10, 0.01)

In [ ]:
randomint = random.randint(0,1000)

In [ ]:
Y_dev.shape

(1000,)

In [ ]:
Y_dev[randomint].shape

()

In [ ]:
X_dev[:,randomint].shape


(784,)

In [ ]:
ee, r, t, out = forward_prop(X_train[:,35].reshape(784, 1), WW1, WW2, BB1, BB2)

In [ ]:
Y_train.shape

In [ ]:
accuracy(A2, Y_train)

In [ ]:
# Z1, Z2, A1, A2 = forward_prop(v, W1, W2, B1, B2)

# def forward_propp(X, W1, W2, B1, B2):
#   Z1 = W1.T.dot(X) + B1
#   A1 = ReLu(Z1)
#   Z2 = W2.T.dot(A1) + B2
#   A2 = Z2
#   return Z1, Z2, A1, A2

In [ ]:
A2